# EO Lab 1 - Multi-Resolution Satellite Image Exploration

**Learning objectives**

By the end of this lab you will be able to:
- Load and display raster imagery from multiple satellite sensors using `rasterio`
- Understand how spatial resolution differs between VHR, Sentinel-2 and Landsat
- Clip rasters to a common spatial extent and visualise them on a shared coordinate grid
- Apply basic image enhancement (percentile stretch, greyscale, false colour)
- Compute spectral indices (NDVI, NDWI) and export results as GeoTIFF for use in QGIS

**Data** -- all files are in EPSG:32635 (UTM zone 35N), covering Istanbul:

| Sensor | Resolution | Bands available |
|---|---|---|
| VHR (very high resolution) | ~1 m | RGB |
| Sentinel-2 | 10 m | Blue B02, Green B03, Red B04, NIR B08 |
| Landsat | 30 m | Blue, Green, Red, NIR |


## 1. Imports and file paths


In [1]:
import numpy as np                  # numerical arrays
import matplotlib.pyplot as plt      # plotting
import rasterio                      # read/write geospatial rasters
from rasterio.windows import from_bounds   # spatial windowing
from rasterio.coords import BoundingBox    # lightweight bounding-box helper

# VHR: single multi-band GeoTIFF, bands 1=R 2=G 3=B at ~1 m resolution
VHR_PATH = "data/vhr/istanbul_vhr.tiff"

# Sentinel-2: each spectral band stored as a separate single-band GeoTIFF (10 m)
S2_PATHS = {
    "R": "data/stac_downloads/sentinel2/s2_B04.tif",   # Red
    "G": "data/stac_downloads/sentinel2/s2_B03.tif",   # Green
    "B": "data/stac_downloads/sentinel2/s2_B02.tif",   # Blue
}
S2_NIR_PATH = "data/stac_downloads/sentinel2/s2_B08.tif"  # Near-Infrared

# Landsat: same single-band layout as Sentinel-2, but 30 m resolution
LS_PATHS = {
    "R": "data/stac_downloads/landsat/landsat_red.tif",
    "G": "data/stac_downloads/landsat/landsat_green.tif",
    "B": "data/stac_downloads/landsat/landsat_blue.tif",
}
LS_NIR_PATH = "data/stac_downloads/landsat/landsat_nir08.tif"


## 2. Helper functions


In [2]:
def pstretch(arr, lo=2, hi=98):
    """
    Percentile stretch: rescale pixel values to [0, 1].

    Raw satellite DN values can span a wide range and vary by sensor.
    A 2-98 % stretch clips extreme outliers and maps the bulk of the data
    to the full 0-1 display range, making images look visually balanced.

    Parameters
    ----------
    arr   : np.ndarray  2-D band array (any numeric dtype)
    lo/hi : float       lower/upper percentile cut-offs (default 2 / 98)
    """
    a = arr.astype(np.float32)
    valid = a[np.isfinite(a) & (a > 0)]   # ignore NoData (0 or NaN)
    if valid.size == 0:
        return np.zeros_like(a)
    lo_v, hi_v = np.percentile(valid, [lo, hi])
    return np.clip((a - lo_v) / (hi_v - lo_v + 1e-9), 0, 1)


def load_single(path, bounds):
    """
    Read a single-band raster file, clipped to *bounds*.

    rasterio.windows.from_bounds() converts a geographic bounding box
    into a pixel window (row/col offsets + size) within the raster grid.
    Only the required pixels are loaded -- no need to read the full scene.
    """
    with rasterio.open(path) as src:
        win = from_bounds(*bounds, transform=src.transform)
        return src.read(1, window=win)   # band index 1


def load_rgb(paths, bounds):
    """
    Stack three single-band files (R, G, B) into an H x W x 3 float array
    ready for matplotlib imshow().
    """
    bands = [pstretch(load_single(paths[k], bounds)) for k in ("R", "G", "B")]
    return np.stack(bands, axis=-1)   # axis=-1 -> last dim = colour channel


def load_vhr_rgb(bounds):
    """Like load_rgb() but for the multi-band VHR file (bands 1, 2, 3)."""
    with rasterio.open(VHR_PATH) as src:
        win = from_bounds(*bounds, transform=src.transform)
        data = src.read([1, 2, 3], window=win)
    return np.stack([pstretch(data[i]) for i in range(3)], axis=-1)


def save_geotiff(array, out_path, bounds, ref_path, nodata=np.nan):
    """
    Save a 2-D float array as a georeferenced single-band GeoTIFF.

    The spatial reference (CRS + pixel transform) is derived from ref_path.
    window_transform() converts the pixel window back to an affine transform
    so QGIS can place the file on the map at the correct location.

    Parameters
    ----------
    array    : 2-D np.ndarray  data to save (float32 recommended)
    out_path : str             destination file path (.tif)
    bounds   : BoundingBox     geographic extent that array covers
    ref_path : str             source raster path (provides CRS / transform)
    nodata   : float           NoData marker (default NaN)
    """
    with rasterio.open(ref_path) as ref:
        win = from_bounds(*bounds, transform=ref.transform)
        win_transform = ref.window_transform(win)
        crs = ref.crs

    with rasterio.open(
        out_path, "w",
        driver="GTiff",
        height=array.shape[0],
        width=array.shape[1],
        count=1,
        dtype=rasterio.float32,
        crs=crs,
        transform=win_transform,
        nodata=nodata,
    ) as dst:
        dst.write(array.astype(np.float32), 1)
    print(f"Saved -> {out_path}")


### Challenge 1 - Inspect the metadata

Before visualising, explore the raster files with `rasterio`.
Open each sensor and print: CRS, pixel resolution (`src.res`),
number of bands (`src.count`), and bounding box (`src.bounds`).

Questions:
1. Are all three sensors in the same CRS? Why does that matter for comparison?
2. How many times larger (in area) is a Landsat pixel vs Sentinel-2? vs VHR?
3. Which sensor covers the largest geographic area?


In [ ]:
# YOUR CODE HERE
# Hint: use  with rasterio.open(path) as src:  and inspect src attributes

for name, path in [
    ("VHR",        VHR_PATH),
    ("Sentinel-2", S2_PATHS["R"]),
    ("Landsat",    LS_PATHS["R"]),
]:
    with rasterio.open(path) as src:
        pass  # replace with print statements


## 3. Full VHR footprint - three sensors side by side


In [ ]:
# The VHR scene is the smallest footprint; all other sensors overlap it.
# Use its bounds as the common clipping window for a fair spatial comparison.
with rasterio.open(VHR_PATH) as src:
    vhr_bounds = src.bounds   # BoundingBox(left, bottom, right, top) in metres

# extent= tells imshow() the real-world coordinate range for axis labels
# format: [left, right, bottom, top]
extent = [vhr_bounds.left, vhr_bounds.right, vhr_bounds.bottom, vhr_bounds.top]

vhr_rgb = load_vhr_rgb(vhr_bounds)
s2_rgb  = load_rgb(S2_PATHS, vhr_bounds)
ls_rgb  = load_rgb(LS_PATHS, vhr_bounds)

# Same geographic area, very different pixel counts -- that is the resolution difference
print(f"VHR  : {vhr_rgb.shape}   (~1 m/px)")
print(f"S2   : {s2_rgb.shape}  (~10 m/px)")
print(f"LS   : {ls_rgb.shape}   (~30 m/px)")


In [ ]:
panels = [
    (vhr_rgb, "VHR  (~1 m)"),
    (s2_rgb,  "Sentinel-2  (~10 m)"),
    (ls_rgb,  "Landsat  (~30 m)"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6),
                          subplot_kw={"aspect": "equal"})
fig.suptitle("Istanbul - same geographic footprint, native resolution",
             fontsize=13, y=1.01)

for ax, (img, title) in zip(axes, panels):
    ax.imshow(img, extent=extent, origin="upper", interpolation="bilinear")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.ticklabel_format(style="plain")

plt.tight_layout()
plt.show()


## 4. Zoom - 2 km x 2 km patch

Viewing the full 8 x 8 km VHR scene makes it hard to appreciate pixel-level
resolution differences. A small central patch makes individual pixels visible,
especially for Landsat.


In [ ]:
# Compute scene centre in UTM metres
cx = (vhr_bounds.left + vhr_bounds.right) / 2
cy = (vhr_bounds.bottom + vhr_bounds.top) / 2
half = 1000   # half-width in metres -> 2 km x 2 km window

zoom_bounds = BoundingBox(cx - half, cy - half, cx + half, cy + half)
zoom_extent = [zoom_bounds.left, zoom_bounds.right,
               zoom_bounds.bottom, zoom_bounds.top]

vhr_z = load_vhr_rgb(zoom_bounds)
s2_z  = load_rgb(S2_PATHS, zoom_bounds)
ls_z  = load_rgb(LS_PATHS, zoom_bounds)

print(f"VHR  zoom : {vhr_z.shape}")
print(f"S2   zoom : {s2_z.shape}")
print(f"LS   zoom : {ls_z.shape}")


### Challenge 2 - Change the zoom area

The current window is centred on the middle of the VHR scene.
Try a different location -- for example the north-west corner,
or an area you think might show interesting features (coastline, park, dense urban).

The VHR scene covers approximately:
`left=662833 m, right=670674 m, bottom=4536673 m, top=4545415 m`

Pick any centre point within those bounds. You can also try `half=500` for a 1 km x 1 km crop.


In [ ]:
# YOUR CODE HERE
# Define zoom_bounds_custom and load vhr_custom, s2_custom, ls_custom
# Then plot them side by side.


## 5. Grayscale


In [ ]:
def to_grey(rgb):
    """
    Convert H x W x 3 RGB to greyscale using ITU-R BT.601 luminance weights.

    Human eyes are most sensitive to green, less to red, and least to blue.
    The weighted sum reflects perceptual sensitivity:
        Y = 0.299 R + 0.587 G + 0.114 B
    """
    return 0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2]


fig, axes = plt.subplots(1, 3, figsize=(18, 6),
                          subplot_kw={"aspect": "equal"})
fig.suptitle("Istanbul - 2 km x 2 km zoom  |  Grayscale",
             fontsize=13, y=1.01)

for ax, (img, title) in zip(axes, [
    (to_grey(vhr_z), "VHR  (~1 m)"),
    (to_grey(s2_z),  "Sentinel-2  (~10 m)"),
    (to_grey(ls_z),  "Landsat  (~30 m)"),
]):
    # interpolation='none' keeps pixels sharp -- important when comparing resolutions
    ax.imshow(img, extent=zoom_extent, origin="upper",
              cmap="gray", interpolation="none")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.ticklabel_format(style="plain")

plt.tight_layout()
plt.show()


## 6. False colour (VNIR)

In a standard false colour composite, the NIR band is mapped to the red channel,
Red to green, and Green to blue. Healthy vegetation -- which strongly reflects NIR --
appears bright red, making it easy to distinguish from built-up areas and bare soil.

VHR only has visible bands (RGB) and is shown as true colour for reference.


In [ ]:
# NIR-Red-Green composites for S2 and Landsat
s2_fc = np.stack([
    pstretch(load_single(S2_NIR_PATH,   zoom_bounds)),   # NIR  -> red channel
    pstretch(load_single(S2_PATHS["R"], zoom_bounds)),   # Red  -> green channel
    pstretch(load_single(S2_PATHS["G"], zoom_bounds)),   # Green-> blue channel
], axis=-1)

ls_fc = np.stack([
    pstretch(load_single(LS_NIR_PATH,   zoom_bounds)),
    pstretch(load_single(LS_PATHS["R"], zoom_bounds)),
    pstretch(load_single(LS_PATHS["G"], zoom_bounds)),
], axis=-1)

fig, axes = plt.subplots(1, 3, figsize=(18, 6),
                          subplot_kw={"aspect": "equal"})
fig.suptitle("Istanbul - 2 km x 2 km zoom  |  False colour (VNIR)",
             fontsize=13, y=1.01)

for ax, (img, title) in zip(axes, [
    (vhr_z,  "VHR  (~1 m)  - RGB (true colour)"),
    (s2_fc,  "Sentinel-2  (~10 m)  - NIR/R/G"),
    (ls_fc,  "Landsat  (~30 m)  - NIR/R/G"),
]):
    ax.imshow(img, extent=zoom_extent, origin="upper", interpolation="none")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.ticklabel_format(style="plain")

plt.tight_layout()
plt.show()


## 7. True colour zoom


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6),
                          subplot_kw={"aspect": "equal"})
fig.suptitle("Istanbul - 2 km x 2 km zoom  |  True colour",
             fontsize=13, y=1.01)

for ax, (img, title) in zip(axes, [
    (vhr_z, "VHR  (~1 m)"),
    (s2_z,  "Sentinel-2  (~10 m)"),
    (ls_z,  "Landsat  (~30 m)"),
]):
    ax.imshow(img, extent=zoom_extent, origin="upper", interpolation="none")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.ticklabel_format(style="plain")

plt.tight_layout()
plt.show()


## 8. Spectral index - NDVI

The Normalised Difference Vegetation Index quantifies vegetation density and health:

    NDVI = (NIR - Red) / (NIR + Red)

Values range from -1 to +1:
- > 0.3  : healthy, dense vegetation
- 0.1-0.3: sparse vegetation / grassland
- ~0     : bare soil, rock
- < 0    : water, cloud, snow

A shared colour scale derived from the 2-98 % percentile of the actual data
is used so both sensors are directly comparable.


In [ ]:
def compute_ndvi(nir_path, red_path, bounds):
    """
    Compute NDVI from separate NIR and Red band files.

    np.errstate suppresses divide-by-zero warnings where NIR + Red == 0.
    np.where() replaces those pixels with NaN so they are transparent
    in plots and excluded from statistics.
    """
    nir = load_single(nir_path, bounds).astype(np.float32)
    red = load_single(red_path, bounds).astype(np.float32)
    with np.errstate(invalid="ignore", divide="ignore"):
        idx = np.where((nir + red) > 0, (nir - red) / (nir + red), np.nan)
    return idx


s2_ndvi = compute_ndvi(S2_NIR_PATH, S2_PATHS["R"], zoom_bounds)
ls_ndvi = compute_ndvi(LS_NIR_PATH, LS_PATHS["R"], zoom_bounds)

# Shared colour scale from combined distribution of both sensors
combined = np.concatenate([
    s2_ndvi[np.isfinite(s2_ndvi)],
    ls_ndvi[np.isfinite(ls_ndvi)],
])
vmin, vmax = np.percentile(combined, [2, 98])
print(f"Shared NDVI colour scale: {vmin:.3f} - {vmax:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 6),
                          subplot_kw={"aspect": "equal"})
fig.suptitle("Istanbul - 2 km x 2 km zoom  |  VHR RGB  &  NDVI",
             fontsize=13, y=1.01)

axes[0].imshow(vhr_z, extent=zoom_extent, origin="upper", interpolation="none")
axes[0].set_title("VHR  (~1 m)  - RGB", fontsize=11)

im1 = axes[1].imshow(s2_ndvi, extent=zoom_extent, origin="upper",
                      interpolation="none", cmap="RdYlGn", vmin=vmin, vmax=vmax)
axes[1].set_title("Sentinel-2  (~10 m)  - NDVI", fontsize=11)
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04, label="NDVI")

im2 = axes[2].imshow(ls_ndvi, extent=zoom_extent, origin="upper",
                      interpolation="none", cmap="RdYlGn", vmin=vmin, vmax=vmax)
axes[2].set_title("Landsat  (~30 m)  - NDVI", fontsize=11)
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04, label="NDVI")

for ax in axes:
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.ticklabel_format(style="plain")

plt.tight_layout()
plt.show()


### Challenge 3 - Interpret NDVI differences between sensors

1. Which areas have the highest NDVI values? Can you identify them in the VHR image?
2. The S2 and Landsat patterns broadly agree but differ in detail. What causes this?
3. Why can we not compute NDVI for VHR? What would we need?
4. Optional: modify compute_ndvi() to mask pixels where NIR < 100. Does it change the map?


## 9. Final challenge - NDWI and export to GeoTIFF

The Normalised Difference Water Index (McFeeters, 1996) highlights open water:

    NDWI = (Green - NIR) / (Green + NIR)

| NDWI value | Surface type |
|---|---|
| > 0.3  | Open water (sea, lake, river) |
| 0-0.3  | Flooded soil / shallow water |
| < 0    | Vegetation, built-up, bare soil |

For Sentinel-2: Green = B03, NIR = B08
For Landsat:    Green = green band, NIR = nir08 band

Tasks:
1. Complete the compute_ndwi() function (one line of maths, same pattern as NDVI).
2. Compute NDWI for both sensors over zoom_bounds.
3. Plot VHR RGB + S2 NDWI + Landsat NDWI side by side using cmap='Blues_r'.
   Derive the colour scale from the data (same technique as NDVI).
4. Save outputs as GeoTIFFs using save_geotiff() and open them in QGIS.

Hint for saving: save_geotiff(array, path, bounds, ref_path)
Use S2_NIR_PATH as ref_path for S2, and LS_NIR_PATH for Landsat.


In [ ]:
import os
os.makedirs("data/outputs", exist_ok=True)


def compute_ndwi(green_path, nir_path, bounds):
    """
    Compute NDWI = (Green - NIR) / (Green + NIR).

    Parameters
    ----------
    green_path : str  path to the Green band file
    nir_path   : str  path to the NIR band file
    bounds     : BoundingBox
    """
    green = load_single(green_path, bounds).astype(np.float32)
    nir   = load_single(nir_path,   bounds).astype(np.float32)
    with np.errstate(invalid="ignore", divide="ignore"):
        # TODO 1: write the NDWI formula here (replace the line below)
        idx = np.full_like(green, np.nan)
    return idx


# TODO 2: compute NDWI for both sensors
s2_ndwi = None   # replace with compute_ndwi(...)
ls_ndwi = None   # replace with compute_ndwi(...)


# TODO 3: plot VHR RGB + S2 NDWI + Landsat NDWI
# Adapt the NDVI plotting block above; use cmap='Blues_r'


# TODO 4: save to GeoTIFF
# save_geotiff(s2_ndwi, "data/outputs/s2_ndwi.tif", zoom_bounds, S2_NIR_PATH)
# save_geotiff(ls_ndwi, "data/outputs/ls_ndwi.tif", zoom_bounds, LS_NIR_PATH)
